[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C04_AI_Agents_Course/01_tool_use/01_tool_use.ipynb)

# 模块 01 · Tool Use 与 function calling

配套讲解：`01_讲解.html`（建议先读完第 1–4 节再动手）。

**学习目标**：
1. 亲手定义 3 个真实工具函数 + 它们的 JSON Schema；
2. 实现**双路径**工具调用：(a) OpenAI 原生 `tools` 参数；(b) 本地小模型 prompted tool calling；
3. 写出统一的 `tool_loop` 多轮循环（含错误回喂与 `max_turns` 保险丝）；
4. 跑一个最小可靠性实验：度量**调用格式正确率**与**参数正确率**（为模块 06 埋线）。

**运行环境**（CPU 即可，全程不需要 GPU）：

| 情况 | 走的路径 |
|---|---|
| 设置了 `OPENAI_API_KEY` | 路径 (a)：openai 客户端原生 function calling |
| 没有 key，但装了 `transformers` | 路径 (b)：`Qwen/Qwen2.5-1.5B-Instruct` prompted tool calling（首次下载约 3GB，CPU 每次生成 10–60 秒） |
| 都没有 / 想跳过下载 | 自动回退到脚本化 **mock**，完整演示协议流程 |

> 本 notebook 的任何 cell 都不会因为缺 key 或缺模型而崩溃。

In [ ]:
import os, json, re, ast, random

HAS_OPENAI_KEY = bool(os.environ.get("OPENAI_API_KEY"))

try:
    import transformers  # noqa: F401
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

USE_MOCK = False   # 设为 True 可强制跳过 API / 本地模型，用脚本化 mock 演示完整流程

print(f"OPENAI_API_KEY 检测: {HAS_OPENAI_KEY}")
print(f"transformers 可用:   {HAS_TRANSFORMERS}")

## 1 · 定义三个真实工具

工具就是普通 Python 函数，但有三条纪律（与讲解第 5/6 节对应）：

- **绝不使用 `eval`**：`calculator` 用 `ast.parse` + 白名单节点遍历做安全求值——这是把"不可信文本"变成"可执行计算"的唯一正当姿势；
- **永远返回字符串**：工具结果最终要回填进模型上下文，统一为文本；
- **失败时返回 `ERROR: ...` 而不是抛异常**：错误信息本身是要回喂给模型的修复线索，要写得具体、可操作。

In [ ]:
import operator

# ---------- 工具 1: calculator —— 安全算术求值（禁 eval） ----------
_BINOPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
           ast.Mod: operator.mod, ast.Pow: operator.pow}
_UNARY = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def _safe_eval(node):
    # 白名单 AST 遍历：只允许数字常量 + 基本算术运算符，其余一律拒绝
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _BINOPS:
        left, right = _safe_eval(node.left), _safe_eval(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 64:
            raise ValueError("指数过大，拒绝计算")
        return _BINOPS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY:
        return _UNARY[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"不允许的语法节点: {type(node).__name__}")

def calculator(expression: str) -> str:
    # 对算术表达式做精确求值；任何失败都以 ERROR: 文本返回（供回喂）
    try:
        return str(_safe_eval(ast.parse(str(expression), mode="eval")))
    except Exception as e:
        return f"ERROR: 无法计算 {expression!r} — {e}"

# ---------- 工具 2: get_word_info —— 内置小词典 ----------
_WORD_DB = {
    "ephemeral": {"pos": "adj.", "definition": "短暂的，转瞬即逝的", "example": "Fame is often ephemeral."},
    "agent":     {"pos": "n.",   "definition": "代理；能感知环境并采取行动的主体", "example": "An LLM agent can call tools."},
    "robust":    {"pos": "adj.", "definition": "健壮的，鲁棒的", "example": "We need a robust JSON parser."},
    "schema":    {"pos": "n.",   "definition": "模式；数据的结构定义", "example": "Each tool has a JSON schema."},
    "latency":   {"pos": "n.",   "definition": "延迟", "example": "Parallel calls reduce latency."},
    "evaluate":  {"pos": "v.",   "definition": "评估，求值", "example": "We evaluate tool-calling reliability."},
}

def get_word_info(word: str) -> str:
    entry = _WORD_DB.get(str(word).lower().strip())
    if entry is None:
        return f"ERROR: 词典中没有 {word!r}。收录的词: {sorted(_WORD_DB)}"
    return json.dumps(entry, ensure_ascii=False)

# ---------- 工具 3: unit_convert —— 单位换算 ----------
_LENGTH_TO_M = {"m": 1.0, "km": 1000.0, "cm": 0.01, "mile": 1609.344, "ft": 0.3048}
_WEIGHT_TO_KG = {"kg": 1.0, "g": 0.001, "lb": 0.45359237}

def unit_convert(value, from_unit: str, to_unit: str) -> str:
    try:
        v = float(value)
    except (TypeError, ValueError):
        return f"ERROR: value 必须是数字，收到 {value!r}"
    f, t = str(from_unit), str(to_unit)
    if f in _LENGTH_TO_M and t in _LENGTH_TO_M:
        return str(round(v * _LENGTH_TO_M[f] / _LENGTH_TO_M[t], 6))
    if f in _WEIGHT_TO_KG and t in _WEIGHT_TO_KG:
        return str(round(v * _WEIGHT_TO_KG[f] / _WEIGHT_TO_KG[t], 6))
    if {f, t} == {"celsius", "fahrenheit"}:
        return str(round(v * 9 / 5 + 32, 6)) if f == "celsius" else str(round((v - 32) * 5 / 9, 6))
    return (f"ERROR: 不支持 {f} -> {t}。支持: {sorted(_LENGTH_TO_M)} / "
            f"{sorted(_WEIGHT_TO_KG)} / celsius<->fahrenheit")

# ---------- 工具注册表（registry）：名字 -> 可调用对象 ----------
TOOLS = {"calculator": calculator, "get_word_info": get_word_info, "unit_convert": unit_convert}

print(calculator("127 * 893"))
print(get_word_info("ephemeral"))
print(unit_convert(42, "km", "mile"))
print(calculator("__import__('os')"))   # 注入尝试会被白名单拒绝

## 2 · 为每个工具写 JSON Schema

模型"看到"的工具只有 schema。注意三处接口设计（讲解第 5 节）：

- `description` 写清**何时调用**，不只是"做什么"；
- 单位参数用 **`enum`** 锁死取值空间——模型不可能拼错枚举值的语义；
- `required` 只包含模型必须从用户话语中获得的信息。

> 这里用 OpenAI 的 `parameters` 字段名；Anthropic 对应字段叫 `input_schema`，schema 本体完全一样。

In [ ]:
TOOL_SCHEMAS = [
    {
        "name": "calculator",
        "description": "对算术表达式做精确求值（支持 + - * / // % ** 和括号）。"
                       "当问题需要数值计算时调用此工具，不要心算。",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string",
                               "description": "纯算术表达式，例如 '127 * 893'，不要包含等号或文字"}
            },
            "required": ["expression"],
        },
    },
    {
        "name": "get_word_info",
        "description": "查询内置词典中英文单词的词性、释义与例句。当用户询问某个英文单词的含义时调用。",
        "parameters": {
            "type": "object",
            "properties": {
                "word": {"type": "string", "description": "要查询的英文单词（小写）"}
            },
            "required": ["word"],
        },
    },
    {
        "name": "unit_convert",
        "description": "在长度(m/km/cm/mile/ft)、重量(kg/g/lb)、温度(celsius/fahrenheit)单位之间换算数值。"
                       "当用户给出 'X 公里是多少英里' 这类问题时调用。",
        "parameters": {
            "type": "object",
            "properties": {
                "value": {"type": "number", "description": "待换算的数值"},
                "from_unit": {"type": "string",
                              "enum": ["m", "km", "cm", "mile", "ft", "kg", "g", "lb",
                                       "celsius", "fahrenheit"]},
                "to_unit": {"type": "string",
                            "enum": ["m", "km", "cm", "mile", "ft", "kg", "g", "lb",
                                     "celsius", "fahrenheit"]},
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    },
]
SCHEMA_BY_NAME = {s["name"]: s for s in TOOL_SCHEMAS}

print(json.dumps(TOOL_SCHEMAS[2], ensure_ascii=False, indent=2))

## 3 · 双路径 LLM 后端

把"模型这一侧"抽象成统一接口：`llm.step(events) -> action`。

- `events` 是与厂商无关的事件流：`("user", 问题)` / `("tool_call", {...})` / `("tool_result", {...})`；
- `action` 要么是 `{"type": "tool_call", "name", "args", "id"}`，要么是 `{"type": "final", "text"}`。

三个实现：

| 类 | 路径 | 说明 |
|---|---|---|
| `OpenAINativeLLM` | (a) 原生 | `tools=` 参数 → 解析 `tool_calls` → `role="tool"` 回填 |
| `QwenPromptedLLM` | (b) prompted | system prompt 约定输出 ```json {"tool":..., "args":...}```，自己解析 |
| `MockLLM` | 回退 | 关键词规则脚本，零依赖演示协议流程 |

先给出 prompted 路线需要的**基础版鲁棒提取器** `parse_tool_call`（剥围栏 → 花括号深度配对 → `json.loads`）。它故意不处理单引号等更脏的情况——那是 ✏️ 练习 1 的任务。

In [ ]:
def parse_tool_call(text: str):
    # 基础版鲁棒提取：容忍 markdown 围栏与前后自然语言"废话"
    # 返回 {"tool": str, "args": dict} 或 None
    if not isinstance(text, str):
        return None
    cleaned = re.sub(r"```(?:json)?", "", text)          # 1) 剥掉围栏标记
    for start in [i for i, ch in enumerate(cleaned) if ch == "{"]:
        depth = 0                                         # 2) 花括号深度配对（正确处理嵌套）
        for end in range(start, len(cleaned)):
            if cleaned[end] == "{":
                depth += 1
            elif cleaned[end] == "}":
                depth -= 1
                if depth == 0:
                    try:
                        obj = json.loads(cleaned[start:end + 1])   # 3) 严格 JSON 解析
                    except json.JSONDecodeError:
                        obj = None
                    if isinstance(obj, dict) and "tool" in obj:
                        return {"tool": obj["tool"], "args": obj.get("args", {})}
                    break                                 # 此起点失败 → 试下一个 '{'
    return None


def build_system_prompt(schemas):
    tools_desc = json.dumps(schemas, ensure_ascii=False, indent=2)
    return ("你是一个可以调用工具的助手。可用工具（JSON Schema）：\n" + tools_desc +
            "\n\n规则：\n"
            "1. 需要调用工具时，只输出一个 JSON 对象，并用 markdown 围栏包裹：\n"
            '```json\n{"tool": "<工具名>", "args": {<参数>}}\n```\n'
            "2. 看到以 [工具结果] 开头的消息后，若信息已足够，直接用中文给出最终答案，不要再输出 JSON。\n"
            "3. 绝不编造不存在的工具或参数。")


class OpenAINativeLLM:
    # 路径 (a)：OpenAI 原生 function calling
    name = "openai-native"

    def __init__(self, schemas, model="gpt-4o-mini"):
        from openai import OpenAI                # openai >= 1.x 客户端风格
        self.client = OpenAI()
        self.model = model
        self.native_tools = [{"type": "function", "function": s} for s in schemas]

    def _to_messages(self, events):
        msgs = []
        for kind, data in events:
            if kind == "user":
                msgs.append({"role": "user", "content": data})
            elif kind == "tool_call":            # assistant 的调用块必须原样放回历史
                msgs.append({"role": "assistant", "content": None, "tool_calls": [{
                    "id": data["id"], "type": "function",
                    "function": {"name": data["name"],
                                 "arguments": json.dumps(data["args"], ensure_ascii=False)}}]})
            elif kind == "tool_result":          # 结果用 role="tool" + tool_call_id 配对
                msgs.append({"role": "tool", "tool_call_id": data["id"],
                             "content": str(data["content"])})
        return msgs

    def step(self, events):
        resp = self.client.chat.completions.create(
            model=self.model, messages=self._to_messages(events), tools=self.native_tools)
        msg = resp.choices[0].message
        if msg.tool_calls:                       # finish_reason == "tool_calls"
            if len(msg.tool_calls) > 1:
                print(f"  (模型发起了 {len(msg.tool_calls)} 个并行调用，本演示只处理第一个)")
            tc = msg.tool_calls[0]
            try:
                args = json.loads(tc.function.arguments)   # arguments 是 JSON 字符串！
            except json.JSONDecodeError:
                args = {}
            return {"type": "tool_call", "id": tc.id, "name": tc.function.name, "args": args}
        return {"type": "final", "text": msg.content or ""}


class QwenPromptedLLM:
    # 路径 (b)：本地小模型 prompted tool calling（默认路径）
    name = "qwen-prompted"
    MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

    def __init__(self, schemas):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        print(f"加载 {self.MODEL_ID}（首次下载约 3GB；CPU 每次生成约 10-60 秒）...")
        self.tok = AutoTokenizer.from_pretrained(self.MODEL_ID)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.MODEL_ID, torch_dtype="auto", device_map="auto")
        self.system = build_system_prompt(schemas)

    def _generate(self, chat_messages, temperature=None):
        text = self.tok.apply_chat_template(chat_messages, tokenize=False,
                                            add_generation_prompt=True)
        inputs = self.tok([text], return_tensors="pt").to(self.model.device)
        kwargs = {"max_new_tokens": 256}
        if temperature is None:
            kwargs["do_sample"] = False          # 演示用贪心，轨迹可复现
        else:
            kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
        out = self.model.generate(**inputs, **kwargs)
        return self.tok.decode(out[0][inputs["input_ids"].shape[1]:],
                               skip_special_tokens=True)

    def _to_messages(self, events):
        msgs = [{"role": "system", "content": self.system}]
        for kind, data in events:
            if kind == "user":
                msgs.append({"role": "user", "content": data})
            elif kind == "tool_call":
                payload = json.dumps({"tool": data["name"], "args": data["args"]},
                                     ensure_ascii=False)
                msgs.append({"role": "assistant", "content": "```json\n" + payload + "\n```"})
            elif kind == "tool_result":          # prompted 路线没有专用角色，用 user 回喂
                msgs.append({"role": "user", "content":
                             f"[工具结果] {data['name']} 返回: {data['content']}\n"
                             "若信息已足够，请直接给出最终答案。"})
        return msgs

    def step(self, events):
        raw = self._generate(self._to_messages(events))
        call = parse_tool_call(raw)
        if call is not None:
            return {"type": "tool_call", "id": "call_local",
                    "name": call["tool"], "args": call["args"]}
        return {"type": "final", "text": raw.strip()}

    def raw_sample(self, question, temperature=0.7):
        # 供可靠性实验使用：对同一问题做带温度采样，返回未解析的原始文本
        msgs = [{"role": "system", "content": self.system},
                {"role": "user", "content": question}]
        return self._generate(msgs, temperature=temperature)


class MockLLM:
    # 零依赖回退：关键词规则模拟"决策"，只演示协议流程，不代表真实模型能力
    name = "mock"

    def step(self, events):
        last_kind, last_data = events[-1]
        if last_kind == "tool_result":
            return {"type": "final",
                    "text": f"(mock 最终答案) 工具 {last_data['name']} 返回: {last_data['content']}"}
        question = events[0][1]
        if "单词" in question or "意思" in question:
            m = re.search(r"[A-Za-z]+", question)
            if m:
                return {"type": "tool_call", "id": "call_mock",
                        "name": "get_word_info", "args": {"word": m.group(0)}}
        if "公里" in question and "英里" in question:
            m = re.search(r"[-+]?\d+(?:\.\d+)?", question)
            if m:
                return {"type": "tool_call", "id": "call_mock", "name": "unit_convert",
                        "args": {"value": float(m.group(0)),
                                 "from_unit": "km", "to_unit": "mile"}}
        m = re.search(r"([\d\.\s\+\-\*/\(\)]+)\s*等于", question)
        if m:
            return {"type": "tool_call", "id": "call_mock", "name": "calculator",
                    "args": {"expression": m.group(1).strip()}}
        return {"type": "final", "text": "(mock) 无法匹配规则，直接作答。"}


# ---------- 后端选择（优雅回退，绝不崩溃） ----------
if USE_MOCK:
    llm = MockLLM()
elif HAS_OPENAI_KEY:
    llm = OpenAINativeLLM(TOOL_SCHEMAS)
elif HAS_TRANSFORMERS:
    try:
        llm = QwenPromptedLLM(TOOL_SCHEMAS)      # ⏳ 重型步骤：~3GB 下载 + CPU 推理
    except Exception as e:
        print(f"本地模型加载失败（{e}），回退到 MockLLM")
        llm = MockLLM()
else:
    llm = MockLLM()

print(f"当前后端: {llm.name}")

## 4 · 统一的 `tool_loop`：多轮循环直到最终答案

与讲解第 2/6 节一一对应：

- 循环体 = `llm.step` → 解析动作 → 执行工具 → 把（调用块, 结果）追加进事件流；
- **三类错误都不终止循环**，而是构造成 `ERROR: ...` 文本回喂，给模型修复机会；
- `max_turns` 是防死循环的保险丝——没有它，"失败→原样重试"会无限烧 token。

In [ ]:
def tool_loop(question, tools, llm, max_turns=5, verbose=True):
    # 统一多轮工具调用循环：厂商无关，依赖 llm.step(events) 抽象
    events = [("user", question)]
    if verbose:
        print(f"❓ {question}")
    for turn in range(max_turns):
        action = llm.step(events)
        if action["type"] == "final":
            if verbose:
                print(f"✅ 最终答案: {action['text']}")
            return action["text"]
        name, args = action["name"], action.get("args") or {}
        call_id = action.get("id", f"call_{turn}")
        if verbose:
            print(f"  → [turn {turn}] 调用 {name}({json.dumps(args, ensure_ascii=False)})")
        if name not in tools:                    # 故障③ 幻觉工具 → 回喂可用清单
            result = f"ERROR: 工具 {name!r} 不存在。可用工具: {sorted(tools)}"
        else:
            try:
                result = str(tools[name](**args))
            except TypeError as e:               # 故障① 参数不匹配 → 回喂
                result = f"ERROR: 参数不匹配 — {e}"
            except Exception as e:               # 故障② 执行异常 → 回喂
                result = f"ERROR: 执行失败 — {type(e).__name__}: {e}"
        if verbose:
            print(f"  ← 结果: {result}")
        events.append(("tool_call", {"id": call_id, "name": name, "args": args}))
        events.append(("tool_result", {"id": call_id, "name": name, "content": result}))
    if verbose:
        print("⚠️ 达到 max_turns 上限，强制结束")
    return "(未完成: 达到 max_turns)"


# ---------- 跑 3 个需要工具的问题，观察完整轨迹 ----------
questions = [
    "127 * 893 等于多少？",
    "英文单词 ephemeral 是什么意思？",
    "42 公里等于多少英里？",
]
for q in questions:
    tool_loop(q, TOOLS, llm)
    print("-" * 60)

## 5 · 可靠性小实验：同一问题跑 10 次

度量两层指标（讲解第 7 节）：

- **调用格式正确率**：原始输出能否被 `parse_tool_call` 解析成合法调用；
- **参数正确率**：解析成功的调用中，工具名与参数是否语义正确。

按后端采样方式不同：Qwen 路径用 `temperature=0.7` 采样 10 次真实输出；OpenAI 路径调 10 次 API；
mock 回退时**合成 10 条带受控噪声的输出**（围栏 / 废话 / 单引号 / 拒绝输出 JSON）——数字是假的，但**度量流程与真实场景完全一致**。

> 注意 $N=10$ 时标准误最高可达 0.16，只能看趋势；正式评测需要 $N$ 上百并报置信区间。

In [ ]:
N_RUNS = 10
QUESTION = "127 * 893 等于多少？"

def sample_raw_outputs(n):
    # 返回 n 条"模型第一轮原始输出"文本
    if isinstance(llm, QwenPromptedLLM):
        return [llm.raw_sample(QUESTION, temperature=0.7) for _ in range(n)]   # ⏳ CPU 上约 2-10 分钟
    if isinstance(llm, OpenAINativeLLM):
        outs = []
        for _ in range(n):
            act = llm.step([("user", QUESTION)])
            if act["type"] == "tool_call":       # 原生路径格式由 API 保证，这里序列化回文本以统一度量
                outs.append(json.dumps({"tool": act["name"], "args": act["args"]},
                                       ensure_ascii=False))
            else:
                outs.append(act["text"])
        return outs
    # mock 回退：合成带随机噪声的输出，演示同一套度量流程
    rng = random.Random(42)
    base = '{"tool": "calculator", "args": {"expression": "127*893"}}'
    corruptions = [
        lambda s: s,                                    # 干净 JSON
        lambda s: "```json\n" + s + "\n```",            # markdown 围栏
        lambda s: "好的，我来计算。" + s + " 请稍等。",    # 前后废话
        lambda s: s.replace('"', "'"),                  # 单引号伪 JSON（基础版解析会失败）
        lambda s: "答案大概是 113411。",                  # 拒绝输出 JSON
    ]
    return [rng.choice(corruptions)(base) for _ in range(n)]

raw_outputs = sample_raw_outputs(N_RUNS)
parsed = [parse_tool_call(t) for t in raw_outputs]
fmt_ok  = [p is not None for p in parsed]
args_ok = [bool(p) and p["tool"] == "calculator"
           and "expression" in (p.get("args") or {}) for p in parsed]

fmt_rate, args_rate = sum(fmt_ok) / N_RUNS, sum(args_ok) / N_RUNS
se = (fmt_rate * (1 - fmt_rate) / N_RUNS) ** 0.5
print(f"后端: {llm.name}   N = {N_RUNS}")
print(f"调用格式正确率: {fmt_rate:.0%}   (SE ≈ {se:.2f})")
print(f"参数正确率:     {args_rate:.0%}")
print("--- 逐条输出 ---")
for t, ok in zip(raw_outputs, fmt_ok):
    flag = "✓" if ok else "✗"
    print(f"  {flag} {t[:72]!r}")

## ✏️ 练习 1：实现 `extract_tool_call(text)`

基础版 `parse_tool_call` 在可靠性实验里栽在了"单引号伪 JSON"上。请实现一个更强的提取器，
对 5 种混乱输出全部正确处理：

1. markdown 围栏包裹；
2. **单引号**伪 JSON（提示：`json.loads` 失败后尝试 `ast.literal_eval`）；
3. JSON 前后夹杂自然语言废话；
4. JSON 内部有**嵌套对象**（提示：花括号深度配对，不要用非贪婪正则）；
5. 完全没有 JSON → 返回 `None`。

返回格式：`{"tool": <str>, "args": <dict>}` 或 `None`（10–20 行内可完成）。

In [ ]:
def extract_tool_call(text):
    # TODO: 实现满足 5 种用例的鲁棒提取器
    #   提示步骤：
    #   1) re.sub 去掉 ```json / ``` 围栏标记
    #   2) 对每个 '{' 起点做花括号深度配对，截出候选子串
    #   3) 先 json.loads；失败再 ast.literal_eval（处理单引号）
    #   4) 候选对象是 dict 且含 "tool" 键 → 返回 {"tool": ..., "args": obj.get("args", {})}
    #   5) 所有候选都失败 → 返回 None
    raise NotImplementedError("请完成 extract_tool_call")

In [ ]:
# ---------- 练习 1 自测 ----------
case1 = '```json\n{"tool": "calculator", "args": {"expression": "1+1"}}\n```'
case2 = "{'tool': 'calculator', 'args': {'expression': '2*3'}}"
case3 = '好的，我将调用工具：{"tool": "get_word_info", "args": {"word": "agent"}} 请稍候。'
case4 = ('{"tool": "unit_convert", "args": {"value": 5, "from_unit": "km", "to_unit": "m"}, '
         '"note": {"nested": {"ok": 1}}}')
case5 = "我不需要工具，答案是 42。"

assert extract_tool_call(case1) == {"tool": "calculator", "args": {"expression": "1+1"}}
assert extract_tool_call(case2) == {"tool": "calculator", "args": {"expression": "2*3"}}
assert extract_tool_call(case3) == {"tool": "get_word_info", "args": {"word": "agent"}}
r4 = extract_tool_call(case4)
assert r4 is not None and r4["tool"] == "unit_convert" and r4["args"]["value"] == 5
assert extract_tool_call(case5) is None
assert extract_tool_call("") is None          # 边界：空字符串
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `validate_args(args, schema)`

执行前防线（讲解第 6 节故障①）：在真正调用工具**之前**对照 schema 校验参数。
只需支持三种检查（10–20 行内可完成）：

1. **必填项**：`required` 中的字段必须出现在 `args` 里；
2. **类型**：只需支持 `"string"` 与 `"number"`（⚠️ Python 中 `bool` 是 `int` 的子类，`True` 不应通过 number 检查）；
3. **枚举**：属性带 `"enum"` 时，值必须在枚举列表中。

返回 `(ok: bool, message: str)`；失败信息要**指名道姓**（这正是要回喂给模型的修复线索）。

In [ ]:
def validate_args(args, schema):
    # TODO: 按 schema["parameters"] 校验 args
    #   1) required 缺失 -> (False, f"缺少必填参数: {名字}")
    #   2) args 中出现 properties 没有的键 -> (False, "未知参数: ...")
    #   3) 类型检查: "string" -> str; "number" -> int/float 且排除 bool
    #   4) 枚举检查: 值必须 in spec["enum"]
    #   5) 全部通过 -> (True, "ok")
    raise NotImplementedError("请完成 validate_args")

In [ ]:
# ---------- 练习 2 自测 ----------
schema_uc = SCHEMA_BY_NAME["unit_convert"]

ok, msg = validate_args({"value": 42, "from_unit": "km", "to_unit": "mile"}, schema_uc)
assert ok, msg                                                       # 合法调用

ok, msg = validate_args({"value": 42, "from_unit": "km"}, schema_uc)
assert not ok and "to_unit" in msg                                   # 缺必填项，且报错指名道姓

ok, msg = validate_args({"value": "四十二", "from_unit": "km", "to_unit": "mile"}, schema_uc)
assert not ok                                                        # 类型错误: string 当 number

ok, msg = validate_args({"value": True, "from_unit": "km", "to_unit": "mile"}, schema_uc)
assert not ok, "bool 不应通过 number 检查"                            # 边界: bool 陷阱

ok, msg = validate_args({"value": 42, "from_unit": "公里", "to_unit": "mile"}, schema_uc)
assert not ok                                                        # 枚举外取值

print("✅ 练习 2 通过")

## ✏️ 练习 3：给 registry 新增无参工具 `today_weekday()`

现在的 `TOOLS[name](**args)` 隐含假设"工具都有参数"。请：

1. 实现 `today_weekday()`：返回今天是星期几（`WEEKDAYS` 中的英文字符串）；
2. 实现统一分发函数 `dispatch(name, args=None)`：工具不存在 / 参数不匹配时**返回** `ERROR: ...` 字符串（不抛异常）；`args` 为 `None` 或 `{}` 时也能正确调用无参工具；
3. 把 `today_weekday` 注册进 `TOOLS`，并在 `SCHEMA_BY_NAME` 中补一个**无参数 schema**（`properties` 为空 dict，`required` 为空 list）。

In [ ]:
import datetime

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def today_weekday() -> str:
    # TODO: 返回今天是星期几
    # 提示: datetime.date.today().weekday() 返回 0-6（0=Monday）
    raise NotImplementedError

def dispatch(name, args=None):
    # TODO: 统一工具分发
    #   1) name 不在 TOOLS 中 -> 返回 "ERROR: ..."（不要抛异常）
    #   2) args 为 None 时按 {} 处理（提示: **(args or {})）
    #   3) TypeError -> 返回 "ERROR: 参数不匹配 — ..."
    raise NotImplementedError

# TODO: 注册工具与 schema
# TOOLS["today_weekday"] = today_weekday
# SCHEMA_BY_NAME["today_weekday"] = {...}

In [ ]:
# ---------- 练习 3 自测 ----------
assert "today_weekday" in TOOLS, "请先把 today_weekday 注册进 TOOLS"
assert today_weekday() in WEEKDAYS
assert dispatch("today_weekday") in WEEKDAYS              # 不传 args
assert dispatch("today_weekday", {}) in WEEKDAYS          # 空 args
assert dispatch("calculator", {"expression": "2+2"}) == "4"
assert dispatch("no_such_tool", {}).startswith("ERROR")   # 幻觉工具
assert dispatch("calculator", {"wrong_param": 1}).startswith("ERROR")   # 参数不匹配
sch = SCHEMA_BY_NAME.get("today_weekday")
assert sch is not None and sch["parameters"]["properties"] == {}
assert sch["parameters"].get("required", []) == []
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。每题一个 cell。

In [ ]:
# 练习 1 参考实现（先自己做，再对照）
def extract_tool_call(text):
    if not isinstance(text, str):
        return None
    cleaned = re.sub(r"```(?:json)?", "", text)            # 围栏
    for start in [i for i, ch in enumerate(cleaned) if ch == "{"]:
        depth = 0
        for end in range(start, len(cleaned)):             # 深度配对，正确处理嵌套
            if cleaned[end] == "{":
                depth += 1
            elif cleaned[end] == "}":
                depth -= 1
                if depth == 0:
                    candidate = cleaned[start:end + 1]
                    obj = None
                    try:
                        obj = json.loads(candidate)        # 严格 JSON
                    except json.JSONDecodeError:
                        try:
                            obj = ast.literal_eval(candidate)   # 宽松回退：单引号伪 JSON
                        except (ValueError, SyntaxError):
                            obj = None
                    if isinstance(obj, dict) and "tool" in obj:
                        return {"tool": obj["tool"], "args": obj.get("args", {})}
                    break
    return None

In [ ]:
# 练习 2 参考实现（先自己做，再对照）
def validate_args(args, schema):
    params = schema.get("parameters", schema)
    props = params.get("properties", {})
    for req in params.get("required", []):
        if req not in args:
            return False, f"缺少必填参数: {req}"
    for key, val in args.items():
        if key not in props:
            return False, f"未知参数: {key}。合法参数: {sorted(props)}"
        spec = props[key]
        expected = spec.get("type")
        if expected == "string" and not isinstance(val, str):
            return False, f"参数 {key} 应为 string，收到 {type(val).__name__}"
        if expected == "number" and (isinstance(val, bool)
                                     or not isinstance(val, (int, float))):
            return False, f"参数 {key} 应为 number，收到 {type(val).__name__}"
        if "enum" in spec and val not in spec["enum"]:
            return False, f"参数 {key} 取值 {val!r} 不在枚举 {spec['enum']} 中"
    return True, "ok" 

In [ ]:
# 练习 3 参考实现（先自己做，再对照）
import datetime

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def today_weekday() -> str:
    return WEEKDAYS[datetime.date.today().weekday()]

def dispatch(name, args=None):
    if name not in TOOLS:
        return f"ERROR: 工具 {name!r} 不存在。可用工具: {sorted(TOOLS)}"
    try:
        return str(TOOLS[name](**(args or {})))
    except TypeError as e:
        return f"ERROR: 参数不匹配 — {e}"

TOOLS["today_weekday"] = today_weekday
SCHEMA_BY_NAME["today_weekday"] = {
    "name": "today_weekday",
    "description": "返回今天是星期几（英文）。当用户询问今天是星期几时调用。无需任何参数。",
    "parameters": {"type": "object", "properties": {}, "required": []},
}
if SCHEMA_BY_NAME["today_weekday"] not in TOOL_SCHEMAS:
    TOOL_SCHEMAS.append(SCHEMA_BY_NAME["today_weekday"])
print(dispatch("today_weekday"))

## 小结

本模块把 agent 的最小构件——**单次工具调用**——完整走了一遍：

- 工具 = 函数 + JSON Schema；schema 是模型唯一"看得见"的接口，质量直接决定调用正确率；
- 协议四阶段：声明 → 决策 → 执行 → 回填；**模型从不执行工具**，错误以文本回喂形成修复循环；
- 双路径：原生 function calling（格式由 API 保证）vs prompted（格式靠指令遵循 + 鲁棒解析）；
- 可靠性必须分层度量：格式正确率 / 参数正确率 / 任务成功率，并警惕小样本与多步复利效应。

**下一步 → 模块 02《从零写 ReAct Agent》**：把今天的 `tool_loop` 升级成
"Thought → Action → Observation" 的显式推理-行动循环 [Yao 2022]，让模型不仅会调工具，
还会**规划何时调、连环调、并从观察中修正计划**。打开 `../02_react_agent/02_讲解.html` 继续。

---
## 🎯 真实数据胶囊题：真实 GSM8K 算式上的计算器工具调用

工具调用的核心是：解析模型发出的调用、执行、把结果喂回。GSM8K 答案里的 `<<8*3=24>>` 是真实算式步骤。实现一个计算器工具，解析这些真实算式并验证结果。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.ai_agents_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn,headers=None):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p):
        req=urllib.request.Request(url, headers=headers or {})
        open(p,"wb").write(urllib.request.urlopen(req,timeout=40).read())
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def mbpp(n=100):
    p=_f("https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl","mbpp.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]

rows=gsm8k(100)
# 真实算式步骤：答案里的 <<lhs=rhs>>
def extract_calcs(ans):
    return re.findall(r"<<([^=]+)=([^>]+)>>", ans)
ex=extract_calcs(rows[0]["answer"])
print("真实算式步骤示例:", ex[:3])

**练习**：实现安全的 `calc_tool(expr)`：只允许数字和 `+-*/(). ` 字符，返回 eval 结果（float）。用它验证真实 GSM8K 算式的 lhs 计算结果 == 标注的 rhs。

In [ ]:
def calc_tool(expr):
    # TODO: 校验 expr 只含安全字符(数字与 +-*/(). 空格)，否则 raise ValueError；返回 float(eval(expr))
    raise NotImplementedError


In [ ]:
# 自测
assert abs(calc_tool("8*3") - 24) < 1e-9
try:
    calc_tool("__import__('os').system('ls')"); assert False, "应拒绝危险输入"
except ValueError: pass
# 在真实 GSM8K 算式上验证
ok=tot=0
for r in rows[:50]:
    for lhs,rhs in extract_calcs(r["answer"]):
        try:
            got=calc_tool(lhs); exp=float(rhs.replace(",",""))
            tot+=1; ok += abs(got-exp)<1e-6
        except Exception: pass
assert tot>0 and ok/tot > 0.9, f"工具应能正确算出>90%真实算式, 得到{ok}/{tot}"
print(f"计算器工具在真实算式上 {ok}/{tot} 正确 ✓")


### 📖 参考答案

In [ ]:
def calc_tool(expr):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expr): raise ValueError("unsafe")
    return float(eval(expr, {"__builtins__":{}}, {}))
print("✓ 工具调用 = 解析+白名单校验+执行+回填，校验是安全关键")